In [13]:
exp_name = '64x32'
exp_folder_name = 'gmd_aquaplanet'

import climt
from sympl import DataArray, TendencyComponent, AdamsBashforth
from sympl import (
    PlotFunctionMonitor, NetCDFMonitor,
    TimeDifferencingWrapper, UpdateFrequencyWrapper,
    set_constant, get_constant, initialize_numpy_arrays_with_properties
)
import gfs_dynamical_core
from datetime import timedelta
import numpy as np
import torch
import torch.nn as nn
import sys, os, pickle
sys.path.append("..")
from models import DynamicMLP, DynamicMLP_flatten

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
class TorchScaler:
    def __init__(self, mean, std):
        self.mean = mean  # [C]
        self.std = std
    
    def transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return (x - mean) / std
    
    def inverse_transform(self, x):
        mean = self.mean
        std = self.std
        if isinstance(mean, torch.Tensor):
            mean = mean.cpu().numpy()
        if isinstance(std, torch.Tensor):
            std = std.cpu().numpy()
        return x * std + mean

class NNParameterization(TendencyComponent):

    input_properties = {
        'air_temperature': {'dims': ['*', 'mid_levels'], 'units': 'degK'},
        'specific_humidity': {'dims': ['*', 'mid_levels'], 'units': 'kg/kg'},
        'surface_air_pressure': {'dims': ['*'], 'units': 'Pa'},
        'surface_upward_latent_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
        'surface_upward_sensible_heat_flux': {'dims': ['*'], 'units': 'W m^-2'},
    }

    tendency_properties = {
        'air_temperature': {
            'units': 'degK day^-1'
        }
    }
    
    diagnostic_properties = {
        'air_temperature_tendency_from_NN': {
            'dims': ['*', 'mid_levels'],
            'units': 'degK day^-1'
        }
    }

    def __init__(self, **kwargs):
        super(NNParameterization, self).__init__(**kwargs)

        self.IN_FEATURES = 59
        self.OUT_FEATURES = 28

        ckpt = torch.load("./best_model/best_model_trial_0.pth",map_location=device)
        ckpt_norm = torch.load('/projects/sds-lab/Shuochen/climt/gmd_aquaplanet/64x32_normalization.pth', map_location=device)
        self.input_scaler  = TorchScaler(ckpt_norm['X_mean'], ckpt_norm['X_std'])
        self.output_scaler = TorchScaler(ckpt_norm['y_mean'], ckpt_norm['y_std'])

        self.model = DynamicMLP_flatten(self.IN_FEATURES,self.OUT_FEATURES,ckpt["hidden_sizes"]).to(device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()

    def array_call(self, state):

        num_cols, num_levs = state['air_temperature'].shape
        
        tendencies = initialize_numpy_arrays_with_properties(
            self.tendency_properties, state, self.input_properties
        )
        diagnostics = initialize_numpy_arrays_with_properties(
            self.diagnostic_properties, state, self.input_properties
        )
        # ---------------------------------
        # Extract state
        # ---------------------------------
        T = state['air_temperature']        # ()
        q = state['specific_humidity']      # ()
        ps = state['surface_air_pressure']  # ()
        lh = state['surface_upward_latent_heat_flux'] # ()
        sh = state['surface_upward_sensible_heat_flux'] # ()
        # ---------------------------
        # Build NN input []
        # ---------------------------        
        x = np.concatenate([
            T,
            q,
            ps[..., None],
            lh[..., None],
            sh[..., None],
        ], axis=-1)  # ()
        
        # normalize
        if self.input_scaler is not None:
            x = self.input_scaler.transform(x)
            
        # Add batch dimension
        x = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
        # ---------------------------------
        # NN inference
        # ---------------------------------
        with torch.no_grad():
            y = self.model(x)   # ()
        y = y.cpu().numpy().squeeze()
        # y = y / 86400.0
        # print(y.shape)
        # inverse normalize
        if self.output_scaler is not None:
            y = self.output_scaler.inverse_transform(y)

        # --------------------------------------------------
        # PHYSICAL CONSTRAINTS (CRITICAL)
        # --------------------------------------------------
        # 1) remove column-mean heating (energy conservation)
        y = y - y.mean(axis=1, keepdims=True)
        
        # 2) clip to physical range (recommended)
        y = np.clip(y, -10.0, 10.0)  # K/day
            
        # ---------------------------------
        # Map NN output → temperature tendency
        # ---------------------------------
        tendencies['air_temperature'][:] = y[:, :num_levs] #shape -> lon*lat, cols
        diagnostics['air_temperature_tendency_from_NN'][:] = y[:, :num_levs] #shape -> lon*lat, cols

        return tendencies, diagnostics
        
set_constant('stellar_irradiance', value=200, units='W m^-2')
model_time_step = timedelta(minutes=10)
# Create components
convection = climt.EmanuelConvection(tendencies_in_diagnostics=True)
simple_physics = TimeDifferencingWrapper(climt.SimplePhysics())
radiation_step = timedelta(hours=1)
# radiation_lw = UpdateFrequencyWrapper(climt.RRTMGLongwave(), radiation_step)
radiation_sw = UpdateFrequencyWrapper(climt.RRTMGShortwave(), radiation_step)
slab_surface = climt.SlabSurface()
# nn component
nn_component = NNParameterization()
dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, nn_component, convection], number_of_damped_levels=5)
# dycore = gfs_dynamical_core.GFSDynamicalCore([simple_physics, slab_surface, radiation_sw, radiation_lw, convection], number_of_damped_levels=5)

grid = climt.get_grid(nx=64, ny=32)

# load state from checkpoint
checkpoint_path = f'/projects/sds-lab/Shuochen/climt/{exp_folder_name}/{exp_name}_checkpoint.pkl'
if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "rb") as f:
        data = pickle.load(f)
    my_state = data["state"]
# my_state = climt.get_default_state([dycore], grid_state=grid)
# # Set initial/boundary conditions
# latitudes = my_state['latitude'].values
# longitudes = my_state['longitude'].values
# zenith_angle = np.radians(latitudes)
# surface_shape = latitudes.shape
# my_state['zenith_angle'].values = zenith_angle
# my_state['eastward_wind'].values[:] = np.random.randn(
#     *my_state['eastward_wind'].shape)
# my_state['ocean_mixed_layer_thickness'].values[:] = 10
# surf_temp_profile = 290 - (40*np.sin(zenith_angle)**2)
# my_state['surface_temperature'].values = surf_temp_profile

def check_state(state, step):
    for k, v in state.items():

        # Extract raw numpy array
        if hasattr(v, "values"):
            arr = v.values
        elif isinstance(v, np.ndarray):
            arr = v
        else:
            continue  # skip non-numeric entries (time, metadata, etc.)

        # Skip non-numeric dtypes
        if not np.issubdtype(arr.dtype, np.number):
            continue

        if not np.all(np.isfinite(arr)):
            bad = np.sum(~np.isfinite(arr))
            raise RuntimeError(
                f"NaN/Inf in '{k}' at step {step} "
                f"(bad count={bad})"
            )
            
for i in range(10000):
    # print(my_state.keys())
    diag, my_state = dycore(my_state, model_time_step)
    my_state.update(diag)
    my_state['time'] += model_time_step
    check_state(my_state, i)
    
    if i % 100 == 0:
        T_mid = my_state['air_temperature'].values[15].mean()
        print(f"Step {i}, mean T_mid = {T_mid:.2f} K")
        T = my_state['air_temperature'].values
        print(i,T.mean(),T.min(),T.max())
        q = my_state['specific_humidity'].values
        print("q min/max:", q.min(), q.max())

Step 0, mean T_mid = 176.87 K
0 188.48209364656765 156.96244761468557 245.09133264412267
q min/max: 0.0 0.00033384565613777257
Step 100, mean T_mid = 177.15 K
100 188.7708281783875 157.06670725913725 244.95476262679983
q min/max: 0.0 0.00033255724788843735
Step 200, mean T_mid = 177.44 K
200 189.05819569451367 157.20126286332746 245.27013510269617
q min/max: 0.0 0.00033682447635947595
Step 300, mean T_mid = 177.73 K
300 189.34482247226904 157.2904062852433 245.45383060574596
q min/max: 0.0 0.0003376833179800273
Step 400, mean T_mid = 178.01 K
400 189.6310811291308 157.41949267412642 245.15973392505825
q min/max: 0.0 0.00033604294152844635
Step 500, mean T_mid = 178.31 K
500 189.9205701558611 157.50708710837668 245.2307989541956
q min/max: 0.0 0.00033357619592357785
Step 600, mean T_mid = 178.61 K
600 190.20344679835694 157.56867350597398 245.1583083908086
q min/max: 0.0 0.00032791192333897804
Step 700, mean T_mid = 178.90 K
700 190.478788460228 157.62827613497788 245.24525549753818
q m

/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/sympl/_core/base_components.py:522: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if properties['units'] is '':
/projects/sds-lab/Shuochen/miniconda3/envs/ai/lib/python3.9/site-packages/sympl/_core/base_components.py:522: SyntaxWarning: "is" with a literal. Did you mean "=="?
  if properties['units'] is '':


KeyboardInterrupt: 